In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [8]:
data_dir = '../notebooks/train' # Use .. to go up one level from 'notebooks'

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,       # Increased from 10
    zoom_range=0.2,          # Increased from 0.1
    width_shift_range=0.15,  # Increased from 0.1
    height_shift_range=0.15, # Increased from 0.1
    shear_range=0.2          # Add shear augmentation
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(28, 28),
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(28, 28),
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 2232 images belonging to 62 classes.
Found 496 images belonging to 62 classes.


In [9]:
from tensorflow.keras.layers import Input # Make sure to import Input

model = Sequential([
    # 1. Add a dedicated Input layer
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5), # Your existing dropout layer
    Dense(train_generator.num_classes, activation='softmax')
])

In [11]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 62)             │         7,998 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 231,742 (905.24 KB)

 Trainable params: 231,742 (905.24 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

# This will monitor the validation loss and stop if it doesn't improve for 3 epochs
early_stopper = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Starting model training...")
history = model.fit(
    train_generator,
    epochs=50, # We can now set a much higher number of epochs
    validation_data=validation_generator,
    callbacks=[early_stopper] # Add the callback here
)
print("Training complete!")

Starting model training...
Epoch 1/50
70/70 ━━━━━━━━━━━━━━━━━━━━ 11s 151ms/step - accuracy: 0.0193 - loss: 4.1355 - val_accuracy: 0.0262 - val_loss: 4.1260
Epoch 2/50
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 145ms/step - accuracy: 0.0175 - loss: 4.1276 - val_accuracy: 0.0222 - val_loss: 4.1226
Epoch 3/50
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 142ms/step - accuracy: 0.0233 - loss: 4.1188 - val_accuracy: 0.0222 - val_loss: 4.0944
Epoch 4/50
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 142ms/step - accuracy: 0.0318 - loss: 4.0600 - val_accuracy: 0.0464 - val_loss: 3.9931
Epoch 5/50
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 141ms/step - accuracy: 0.0578 - loss: 3.8985 - val_accuracy: 0.0887 - val_loss: 3.7191
Epoch 6/50
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 141ms/step - accuracy: 0.0909 - loss: 3.6645 - val_accuracy: 0.1512 - val_loss: 3.4975
Epoch 7/50
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 141ms/step - accuracy: 0.1223 - loss: 3.4446 - val_accuracy: 0.1492 - val_loss: 3.3244
Epoch 8/50
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 143ms/step - accuracy: 0.150

In [14]:
# Cell 6: Save the Trained Model (Updated to the new Keras format)

# The command is the same, we just change the filename's extension
model.save('../ml_models/character_recognition_model.keras')

print("Model saved successfully in the new .keras format!")

Model saved successfully in the new .keras format!
